# 01 — Dataset generation

Generate a small FDTD ground-truth dataset for the three device families used in the paper:
MMI, Y-branch, and directional coupler. This notebook produces a tiny example (~10 geometries
per family at $\lambda = 1.55\,\mu$m) so you can verify the pipeline end-to-end without running
the full 22 500-sample sweep.

**Prerequisites:** `pip install -r requirements.txt` plus Meep — `conda install -c conda-forge pymeep`.

After this notebook, see [`02_training.ipynb`](02_training.ipynb) to train on the resulting shards.

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path('..').resolve()
FDTD_DIR = REPO_ROOT / 'FDTD'
if str(FDTD_DIR) not in sys.path:
    sys.path.insert(0, str(FDTD_DIR))

DATA_ROOT = REPO_ROOT / 'Data' / 'demo_sweep'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print('repo root:', REPO_ROOT)
print('output dir:', DATA_ROOT)

## Run a small unified sweep

`FDTD/unified_sweep.py` is the canonical dataset generator. Here we call it on three families
with 10 geometries each; full Meep simulation per geometry takes a few minutes on CPU.

In [ ]:
import subprocess

cmd = [
    sys.executable, str(FDTD_DIR / 'unified_sweep.py'),
    '--out-dir', str(DATA_ROOT),
    '--n-mmi', '2', '--n-ybranch', '2', '--n-directional-coupler', '2',
    '--wavelengths', '1.55',
    '--n-procs', '4',
    '--shard-size', '8',
]
print(' '.join(cmd))
# subprocess.run(cmd, check=True)   # uncomment to actually run; needs Meep installed


## Inspect a generated sample

Each shard packs many samples; the `index.json` file describes them. Below we load one
directional-coupler sample and plot $\varepsilon_r$, the source mask, and FDTD $|E_z|$.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

shard_dir = DATA_ROOT / 'shards'
if not shard_dir.is_dir():
    print('No shards yet — run the sweep cell above with subprocess uncommented.')
else:
    index = json.loads((shard_dir / 'index.json').read_text())
    entry = next(e for e in index if e['device'] == 'directional_coupler')
    npz = np.load(shard_dir / entry['shard'])
    p = f"s{entry['slot']}/"
    eps  = npz[p + 'eps']
    ezr  = npz[p + 'Ez_real']; ezi = npz[p + 'Ez_imag']
    src  = npz[p + 'src_mask']
    mag  = np.abs(ezr + 1j * ezi)

    fig, axes = plt.subplots(1, 3, figsize=(11, 3), constrained_layout=True)
    axes[0].imshow(eps, origin='lower', cmap='viridis'); axes[0].set_title(r'$\varepsilon_r$')
    axes[1].imshow(src, origin='lower', cmap='Greens');  axes[1].set_title('source mask')
    axes[2].imshow(mag, origin='lower', cmap='magma');   axes[2].set_title(r'FDTD $|E_z|$')
    for a in axes: a.set_xlabel('x px'); a.set_ylabel('y px')
    plt.show()

Once you have a `Data/demo_sweep/shards/index.json`, head to [`02_training.ipynb`](02_training.ipynb) to
smoke-train a model on it.